# 14a - Generate Scenario Evaluation dengan Pipeline RAG dan RAGAS

Notebook Kaggle ini membentuk 20 pertanyaan pada lima skenario, menjalankan pipeline RAG secara self-contained, lalu menilai setiap jawaban dengan RAGAS sebelum menyimpan CSV. Pipeline mencakup query expansion, dense+BM25 retrieval, reciprocal-rank fusion, reranking, penyusunan sitasi, dan generasi jawaban.

Knowledge base, indeks FAISS, model embedding, reranker, dan dua model fine-tuned GGUF diambil dari Hugging Face. Notebook tidak memakai FastAPI atau server QA RAG di laptop.


## Alur

1. Pasang dependensi dan jalankan Ollama di runtime Kaggle.
2. Unduh data uji, knowledge base, indeks FAISS, embedding/reranker, dan dua GGUF dari Hugging Face.
3. Bentuk 20 pertanyaan yang sama pada lima skenario: faktual, interpretatif, variasi gaya, di luar konteks, dan typo.
4. Jalankan RAG end-to-end untuk `llama3.2-finetuned`, `gemma2-finetuned`, dan `deepseek-flash`.
5. Model yang sedang diuji juga melakukan query expansion, penyusunan sitasi, dan jawaban.
6. Hitung `faithfulness`, `answer_relevancy`, dan `context_relevance` dengan RAGAS.
7. Setelah RAGAS selesai, simpan `scenario_results.csv`, buat tiga lembar evaluasi manusia buta, lalu unggah artefak ke Hugging Face.

Notebook 14b hanya menghitung ringkasan RAGAS dan hasil tiga evaluator. Notebook 21 tetap khusus pengujian integrasi dan efisiensi sistem lokal.

> **Penting:** jika sesi sebelumnya sudah menampilkan `numpy.dtype size changed` atau `No module named langchain_community.chat_models.vertexai`, restart Kaggle session/kernel terlebih dahulu, lalu jalankan notebook dari awal. Sel dependensi membekukan stack RAGAS/LangChain yang kompatibel; modul yang telanjur dimuat oleh kernel lama tidak aman digunakan kembali.


In [ ]:
import importlib.metadata as package_metadata
import importlib.util
import subprocess
import sys

# Jangan memasang ulang NumPy/Pandas/Transformers/Hugging Face bawaan
# runtime Kaggle. Mengganti NumPy ketika kernel sudah aktif dapat merusak ABI
# paket biner dan memunculkan "numpy.dtype size changed".
# Stack RAGAS mengikuti 20-rag-evaluation.ipynb yang diberikan pengguna.
# API collections RAGAS 0.4 tidak memakai impor VertexAI lama.
from packaging.specifiers import SpecifierSet
from packaging.version import Version

versioned_packages = {
    # module: (distribution, version_constraint, pip_requirement)
    "ragas": ("ragas", ">=0.4,<0.5", "ragas>=0.4,<0.5"),
    "langchain_community": ("langchain-community", ">=0.3,<0.4.2", "langchain-community>=0.3,<0.4.2"),
    "openai": ("openai", ">=1.50,<2", "openai>=1.50,<2"),
    "instructor": ("instructor", ">=1.3,<1.7", "instructor>=1.3,<1.7"),
    "rank_bm25": ("rank-bm25", "==0.2.2", "rank-bm25==0.2.2"),
}
unversioned_packages = {
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "einops": "einops",
}


def installed_version(distribution):
    try:
        return package_metadata.version(distribution)
    except package_metadata.PackageNotFoundError:
        return None


missing_packages = [
    requirement
    for module, (distribution, constraint, requirement) in versioned_packages.items()
    if (
        importlib.util.find_spec(module) is None
        or installed_version(distribution) is None
        or Version(installed_version(distribution)) not in SpecifierSet(constraint)
    )
]
missing_packages.extend(
    package for module, package in unversioned_packages.items()
    if importlib.util.find_spec(module) is None
)
if missing_packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        *missing_packages,
    ])

# Gagal lebih awal sebelum generasi mahal jika API RAGAS referensi tidak tersedia.
# Jika sesi sebelumnya sudah mengimpor RAGAS lama, restart kernel lalu Run All.
import ragas
from ragas.embeddings import OpenAIEmbeddings as _RagasOpenAIEmbeddings
from ragas.llms import llm_factory as _ragas_llm_factory
from ragas.metrics.collections import AnswerRelevancy as _AnswerRelevancy

import hashlib
import importlib.metadata as metadata
import json
import os
import random
import re
import time
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from huggingface_hub import HfApi, hf_hub_download

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("ragas:", metadata.version("ragas"))
print("langchain-community:", metadata.version("langchain-community"))
print("openai:", metadata.version("openai"))
print("instructor:", metadata.version("instructor"))


## 1. Jalankan Ollama di runtime Kaggle

In [ ]:
subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

OLLAMA_URL = "http://127.0.0.1:11434"
for _ in range(30):
    try:
        requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).raise_for_status()
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama gagal aktif di runtime notebook.")

## 2. Konfigurasi pipeline

In [ ]:
def get_secret(name, kaggle_name=None, prompt=False):
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(kaggle_name or name)
    except Exception:
        value = os.environ.get(name)
    if not value and prompt:
        value = getpass(f"{name}: ")
    if not value:
        raise RuntimeError(f"Secret {name} tidak ditemukan.")
    return value


HF_TOKEN = get_secret("HF_Faiss")
DEEPSEEK_API_KEY = get_secret("DEEPSEEK_API_KEY", prompt=True)
RUN_RAGAS = True
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", prompt=True) if RUN_RAGAS else None
JUDGE_MODEL = "gpt-4o-mini-2024-07-18"
JUDGE_EMBEDDING = "text-embedding-3-small"

HF_REPO = "Makaareeem/publikasi-rag-finetuning-dataset"
HF_KB_REPO = "Makaareeem/publikasi-rag-knowledge-base"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
EMBEDDING_MODEL_ID = "nomic-ai/nomic-embed-text-v1.5"
RERANKER_MODEL_ID = "madebyaris/rerank-indonesia"

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/kaggle/working" if Path("/kaggle/working").is_dir() else "./scenario_evaluation"))
ASSET_DIR = OUTPUT_DIR / "scenario_rag_assets"
KB_DIR = ASSET_DIR / "kb"
MODEL_DIR = ASSET_DIR / "models"
for directory in [OUTPUT_DIR, ASSET_DIR, KB_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RATER_NAMES = ["evaluator_1", "evaluator_2", "evaluator_3"]
SEED = 42
TOP_K = 5
TOP_K_DENSE = 20
TOP_K_SPARSE = 20
TOP_K_RERANK_CANDIDATES = 20
RRF_K = 60
RERANK_SCORE_THRESHOLD = -1.0
QUERY_PREFIX = "search_query: "
QUERY_EXPANSION_TEMPERATURE = 0.3
QUERY_EXPANSION_MAX_TOKENS = 80
CITATION_MAX_TOKENS = 500
MAX_NEW_TOKENS = 512
NUM_CTX = 8192
REQUEST_TIMEOUT = (10, 900)

QUERY_EXPANSION_SYSTEM_PROMPT = (
    "Tugas Anda: tulis ulang SATU pertanyaan menjadi SATU kalimat pencarian yang lebih deskriptif "
    "untuk sistem pencarian dokumen statistik BPS. ATURAN KETAT — WAJIB DIIKUTI: "
    "1. Output HANYA kalimat hasil tulis ulang. Tanpa kata pembuka, basa-basi, tanda kutip, atau penjelasan. "
    "2. JANGAN menjawab pertanyaannya dan JANGAN menyebutkan angka atau fakta spesifik. "
    "3. JANGAN berkomentar tentang teks atau dokumen. 4. Maksimal satu kalimat. "
    "Boleh tambahkan istilah atau singkatan resmi BPS yang relevan."
)
CITATION_SYSTEM_PROMPT = (
    "Anda membantu merangkum jawaban singkat berdasarkan tiap sumber dokumen statistik BPS yang diberikan. "
    "Untuk SETIAP sumber bernomor, tulis satu kalimat yang menjawab pertanyaan HANYA berdasarkan isi sumber tersebut. "
    "WAJIB sertakan tahun atau periode jika sumber menyebutkannya. Jika sumber tidak relevan, tulis "
    "'Sumber ini tidak membahas pertanyaan tersebut.' Jangan mencampur informasi antar sumber. "
    "Jawab HANYA sebagai JSON: {\"sources\": [{\"index\": 1, \"answer\": \"kalimat\"}]}"
)
SYSTEM_PROMPT = (
    "Anda adalah asisten yang menjawab pertanyaan seputar publikasi statistik BPS berdasarkan konteks yang diberikan. "
    "Kutip bagian konteks yang relevan sebelum menjawab, merujuk sumber HANYA dengan nomor sitasi seperti [1] [2], "
    "dan jangan menyebut ulang judul dokumen dalam kalimat jawaban. HANYA gunakan angka atau fakta yang secara eksplisit "
    "tertulis dalam konteks. JANGAN menambahkan angka dari pengetahuan Anda sendiri. Jika konteks memuat TARGET, RENCANA, "
    "atau PROYEKSI, jelaskan dengan jelas bahwa itu bukan angka realisasi. Jika konteks tidak memuat informasi yang relevan, "
    "katakan dengan jujur bahwa Anda tidak memiliki informasi tersebut dalam dokumen yang tersedia. Jangan mengarang jawaban "
    "atau menyimpulkan sesuatu yang tidak didukung oleh konteks."
)

MODEL_CONFIGS = {
    "llama3.2-finetuned": {
        "gguf_repo": "Makaareeem/llama3.2-finetuned-gguf",
        "gguf_filename": "Llama-3.2-3B-Instruct.Q4_K_M.gguf",
        "ollama_name": "llama3.2-finetuned-scenario-rag",
        "system_role": True,
        "repeat_penalty": 1.0,
    },
    "gemma2-finetuned": {
        "gguf_repo": "Makaareeem/gemma2-finetuned-gguf",
        "gguf_filename": "gemma-2-2b-it.Q4_K_M.gguf",
        "ollama_name": "gemma2-finetuned-scenario-rag",
        "system_role": False,
        "repeat_penalty": 1.0,
    },
}
MODELS_FOR_SCENARIO = list(MODEL_CONFIGS) + ["deepseek-flash"]

## 3. Unduh aset Hugging Face dan registrasikan GGUF

In [ ]:
api = HfApi(token=HF_TOKEN)
ASSET_REVISIONS = {
    "kb": api.repo_info(HF_KB_REPO, repo_type="dataset").sha,
    "embedding": api.repo_info(EMBEDDING_MODEL_ID).sha,
    "reranker": api.repo_info(RERANKER_MODEL_ID).sha,
}

INDEX_PATH = Path(hf_hub_download(
    HF_KB_REPO,
    "faiss_index.bin",
    repo_type="dataset",
    revision=ASSET_REVISIONS["kb"],
    token=HF_TOKEN,
    local_dir=KB_DIR,
))
METADATA_PATH = Path(hf_hub_download(
    HF_KB_REPO,
    "chunk_metadata.jsonl",
    repo_type="dataset",
    revision=ASSET_REVISIONS["kb"],
    token=HF_TOKEN,
    local_dir=KB_DIR,
))

model_manifest = {}
for model_key, config in MODEL_CONFIGS.items():
    revision = api.repo_info(config["gguf_repo"]).sha
    gguf_path = Path(hf_hub_download(
        config["gguf_repo"],
        config["gguf_filename"],
        revision=revision,
        token=HF_TOKEN,
        local_dir=MODEL_DIR / model_key,
    ))
    lines = [f'FROM "{gguf_path.resolve().as_posix()}"']
    if config["system_role"]:
        triple_quote = '"' * 3
        lines.append(f"SYSTEM {triple_quote}{SYSTEM_PROMPT}{triple_quote}")
    lines.extend([
        "PARAMETER temperature 0",
        f"PARAMETER num_predict {MAX_NEW_TOKENS}",
        f"PARAMETER repeat_penalty {config['repeat_penalty']}",
        f"PARAMETER num_ctx {NUM_CTX}",
    ])
    modelfile = MODEL_DIR / f"{model_key}.Modelfile"
    modelfile.write_text("\n".join(lines), encoding="utf-8")
    subprocess.run(["ollama", "create", config["ollama_name"], "-f", str(modelfile)], check=True)
    model_manifest[model_key] = {
        "repo": config["gguf_repo"],
        "filename": config["gguf_filename"],
        "revision": revision,
        "ollama_name": config["ollama_name"],
    }

print(json.dumps(model_manifest, indent=2))

## 4. Bangun pipeline retrieval dan reranking

In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from sentence_transformers import CrossEncoder, SentenceTransformer


class NomicQueryEmbeddings(Embeddings):
    def __init__(self):
        self.model = SentenceTransformer(
            EMBEDDING_MODEL_ID,
            trust_remote_code=True,
            revision=ASSET_REVISIONS["embedding"],
            device="cpu",
        )

    def embed_query(self, text):
        vector = self.model.encode(QUERY_PREFIX + text, normalize_embeddings=True)
        return vector.tolist()

    def embed_documents(self, texts):
        raise NotImplementedError("Dokumen sudah di-embed pada indeks Hugging Face.")


def load_documents(metadata_path):
    documents = []
    with open(metadata_path, "r", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            documents.append(Document(page_content=record["text"], metadata=record))
    return documents


def adapt_faiss_to_langchain(index_path, documents, embeddings):
    raw_index = faiss.read_index(str(index_path))
    docstore_dict = {}
    index_to_docstore_id = {}
    for document in documents:
        faiss_id = int(document.metadata["faiss_id"])
        doc_id = str(faiss_id)
        docstore_dict[doc_id] = document
        index_to_docstore_id[faiss_id] = doc_id
    return FAISS(
        embedding_function=embeddings,
        index=raw_index,
        docstore=InMemoryDocstore(docstore_dict),
        index_to_docstore_id=index_to_docstore_id,
    )


documents = load_documents(METADATA_PATH)
embeddings = NomicQueryEmbeddings()
vectorstore = adapt_faiss_to_langchain(INDEX_PATH, documents, embeddings)
bm25_retriever = BM25Retriever.from_documents(
    documents, preprocess_func=lambda text: text.lower().split()
)
bm25_retriever.k = TOP_K_SPARSE
reranker_model = CrossEncoder(
    RERANKER_MODEL_ID,
    revision=ASSET_REVISIONS["reranker"],
    device="cpu",
)


def reciprocal_rank_fusion(rankings, k=RRF_K):
    scores = {}
    lookup = {}
    for ranking in rankings:
        for rank, document in enumerate(ranking, start=1):
            doc_id = int(document.metadata["faiss_id"])
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            lookup[doc_id] = document
    return [lookup[doc_id] for doc_id in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(queries):
    rankings = []
    for query in queries:
        dense = [doc for doc, _ in vectorstore.similarity_search_with_score(query, k=TOP_K_DENSE)]
        sparse = bm25_retriever.invoke(query)[:TOP_K_SPARSE]
        rankings.extend([dense, sparse])
    return reciprocal_rank_fusion(rankings)[:TOP_K_RERANK_CANDIDATES]


def rerank(question, candidate_documents):
    if not candidate_documents:
        return []
    scores = reranker_model.predict([[question, doc.page_content] for doc in candidate_documents])
    ranked = sorted(
        [(doc, float(score)) for doc, score in zip(candidate_documents, scores)],
        key=lambda pair: pair[1],
        reverse=True,
    )
    return [(doc, score) for doc, score in ranked if score >= RERANK_SCORE_THRESHOLD][:TOP_K]

## 5. Bank pertanyaan skenario yang ditulis manual

Seluruh pertanyaan di bawah ditulis langsung dan dapat dikoreksi sebelum eksperimen. Tidak ada pertanyaan yang diambil dari `test.csv`, dibuat oleh LLM, atau ditulis ulang saat notebook berjalan. LLM hanya digunakan setelah bank pertanyaan dibekukan, yaitu untuk expansion, sitasi, jawaban RAG, dan penilaian RAGAS.


In [ ]:
# Bank pertanyaan dibekukan di sini: 5 skenario × 4 pertanyaan = 20 pertanyaan.
# Tidak ada pertanyaan yang dibuat atau ditulis ulang oleh LLM.
# Perbedaan ragam pertanyaan tercermin langsung pada redaksinya.

SCENARIO_1 = [
    {
        "id": "S1_F01", "topic": "penduduk_lanjut_usia",
        "question": "Berapa persentase penduduk lanjut usia di Indonesia menurut publikasi Statistik Penduduk Lanjut Usia edisi terbaru?",
    },
    {
        "id": "S1_F02", "topic": "kondisi_anak",
        "question": "Anak-anak kecil di Indonesia sekarang kondisinya gimana ya, ada datanya nggak?",
    },
    {
        "id": "S1_F03", "topic": "ipak",
        "question": "Berapa Indeks Perilaku Anti Korupsi (IPAK) nasional pada edisi terakhir? Saya butuh angka pastinya.",
    },
    {
        "id": "S1_F04", "topic": "ikps",
        "question": "Apa yang dimaksud dengan Indeks Khusus Penanganan Stunting (IKPS) menurut publikasi BPS?",
    },
]


SCENARIO_2 = [
    {
        "id": "S2_I01", "subtype": "trend_interpretation", "topic": "tren_ipm",
        "question": "Bagaimana tren Indeks Pembangunan Manusia (IPM) Indonesia dari edisi ke edisi?",
    },
    {
        "id": "S2_I02", "subtype": "trend_interpretation", "topic": "kualitas_hidup",
        "question": "Kualitas hidup orang Indonesia sekarang membaik nggak sih dibanding dulu?",
    },
    {
        "id": "S2_I03", "subtype": "comparative_analysis", "topic": "tren_ikps",
        "question": "Bagaimana capaian penanganan stunting berdasarkan Laporan IKPS antarperiode, membaik atau memburuk?",
    },
    {
        "id": "S2_I04", "subtype": "trend_interpretation", "topic": "definisi_dan_perkembangan_ipm",
        "question": "Apa itu Indeks Pembangunan Manusia (IPM) dan bagaimana perkembangan definisinya antaredisi publikasi?",
    },
]


# Empat redaksi menanyakan makna yang sama; versi formal menjadi baseline.
STYLE_VARIANTS = [
    {
        "id": "S3_V01", "style": "formal",
        "question": "Bagaimana tren Indeks Pembangunan Manusia (IPM) Indonesia dari edisi ke edisi?",
    },
    {
        "id": "S3_V02", "style": "informal",
        "question": "ipm indo makin ke sini makin naik atau turun sih?",
    },
    {
        "id": "S3_V03", "style": "langsung_ringkas",
        "question": "Apa tren utama IPM Indonesia dalam beberapa edisi terakhir?",
    },
    {
        "id": "S3_V04", "style": "akademik",
        "question": "Berdasarkan publikasi BPS antaredisi, bagaimana perkembangan IPM Indonesia dan apa kecenderungan umumnya?",
    },
]


OUT_OF_CONTEXT = [
    {"id": "S4_O01", "type": "far_domain", "question": "Siapa juara Piala Dunia 2022?"},
    {"id": "S4_O02", "type": "far_domain", "question": "Bagaimana cara memasak rendang?"},
    {"id": "S4_O03", "type": "current_external", "question": "Berapa harga saham Apple hari ini?"},
    {
        "id": "S4_O04", "type": "future_bps_period",
        "question": "Berapa nilai IPM Indonesia pada tahun 2030 menurut realisasi resmi BPS?",
    },
]


# Minor = satu gangguan kecil pada satu kata.
# Major = sedikitnya tiga gangguan pada beberapa kata, tetapi maksud masih dikenali.
TYPO_CASES = [
    {
        "id": "S5_T01", "base_question_id": "S1_F01",
        "severity": "minor", "operation": "delete_1_character",
        "question": "Berapa persentase penduduk lanjut usia di Indonesia menurut publikasi Statistik Penduduk Lanjut Usia edsi terbaru?",
    },
    {
        "id": "S5_T02", "base_question_id": "S1_F03",
        "severity": "minor", "operation": "swap_2_characters",
        "question": "Berapa Indeks Perilaku Anti Korupsi (IPAK) nasional pada edisi terkahir? Saya butuh angka pastinya.",
    },
    {
        "id": "S5_T03", "base_question_id": "S1_F02",
        "severity": "major", "operation": "delete+replace+swap_multiple_words",
        "question": "Ank-anak kcil di Indoensia skarang kondisinya gmna ya, ada datanya nggak?",
    },
    {
        "id": "S5_T04", "base_question_id": "S1_F04",
        "severity": "major", "operation": "delete+replace+swap_multiple_words",
        "question": "Apa yang dimaksd dgn Indkes Khuss Penanganan Stuntng (IKPS) mnurut publikasi BPS?",
    },
]


def common_fields(item, subtask_type):
    return {
        "source_subtask_type": subtask_type,
        "out_of_context_type": item.get("type", ""),
        "typo_severity": item.get("severity", ""),
    }


rows = []
for item in SCENARIO_1:
    rows.append({
        "scenario": "1_faktual_berbasis_narasi", "variant": item["id"],
        "question": item["question"], "base_question_id": item["id"],
        "style_group_id": "", "style": "clean", "typo_group_id": "", "typo_operations": "",
        **common_fields(item, "factual_retrieval"),
    })

for item in SCENARIO_2:
    rows.append({
        "scenario": "2_interpretatif_ringan", "variant": item["subtype"],
        "question": item["question"], "base_question_id": item["id"],
        "style_group_id": "", "style": "clean", "typo_group_id": "", "typo_operations": "",
        **common_fields(item, item["subtype"]),
    })

for item in STYLE_VARIANTS:
    style_item = {**item, "topic": "tren_ipm"}
    rows.append({
        "scenario": "3_variasi_gaya_bahasa", "variant": item["style"],
        "question": item["question"], "base_question_id": "S2_I01",
        "style_group_id": "STYLE-IPM-TREND", "style": item["style"],
        "typo_group_id": "", "typo_operations": "",
        **common_fields(style_item, "style_variation"),
    })

for item in OUT_OF_CONTEXT:
    rows.append({
        "scenario": "4_di_luar_konteks", "variant": item["type"],
        "question": item["question"], "base_question_id": item["id"],
        "style_group_id": "", "style": "clean", "typo_group_id": "", "typo_operations": "",
        **common_fields(item, "out_of_context"),
    })

clean_lookup = {item["id"]: item for item in SCENARIO_1}
for item in TYPO_CASES:
    clean = clean_lookup[item["base_question_id"]]
    typo_meta = {**clean, **item}
    rows.append({
        "scenario": "5_kesalahan_ketik", "variant": item["severity"],
        "question": item["question"], "base_question_id": item["base_question_id"],
        "style_group_id": "", "style": "typo",
        "typo_group_id": f"TYPO-{item['base_question_id']}", "typo_operations": item["operation"],
        **common_fields(typo_meta, "typo_factual_retrieval"),
    })

scenario_bank = pd.DataFrame(rows)
scenario_bank.insert(0, "scenario_id", [f"Q{i:03d}" for i in range(1, len(scenario_bank) + 1)])

EXPECTED_COUNTS = {
    "1_faktual_berbasis_narasi": 4,
    "2_interpretatif_ringan": 4,
    "3_variasi_gaya_bahasa": 4,
    "4_di_luar_konteks": 4,
    "5_kesalahan_ketik": 4,
}
assert scenario_bank.groupby("scenario").size().to_dict() == EXPECTED_COUNTS
assert len(scenario_bank) == 20 and scenario_bank["scenario_id"].is_unique
assert scenario_bank["question"].str.strip().ne("").all()
assert scenario_bank.loc[scenario_bank["scenario"].eq("3_variasi_gaya_bahasa"), "style"].eq("formal").sum() == 1
typo_rows = scenario_bank[scenario_bank["scenario"].eq("5_kesalahan_ketik")]
assert typo_rows["typo_severity"].value_counts().to_dict() == {"minor": 2, "major": 2}
assert set(typo_rows["base_question_id"]) == set(clean_lookup)

display(scenario_bank[[
    "scenario_id", "scenario", "variant", "style", "typo_severity",
    "question", "base_question_id", "style_group_id", "typo_group_id",
]])


## 6. Fungsi pipeline RAG dan generasi

In [ ]:
def ollama_chat(model_key, messages, temperature=0, num_predict=MAX_NEW_TOKENS, response_format=None):
    config = MODEL_CONFIGS[model_key]
    payload = {
        "model": config["ollama_name"],
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict,
            "repeat_penalty": config["repeat_penalty"],
            "seed": SEED,
            "num_ctx": NUM_CTX,
        },
    }
    if response_format:
        payload["format"] = response_format
    response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def deepseek_chat(messages, temperature=0, max_tokens=MAX_NEW_TOKENS, json_mode=False):
    payload = {
        "model": "deepseek-flash",
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": messages,
    }
    if json_mode:
        payload["response_format"] = {"type": "json_object"}
    response = requests.post(
        f"{DEEPSEEK_BASE_URL}/chat/completions",
        json=payload,
        headers={"Authorization": f"Bearer {DEEPSEEK_API_KEY}"},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    return response.json()


def model_uses_system_role(model_key):
    return True if model_key == "deepseek-flash" else MODEL_CONFIGS[model_key]["system_role"]


def model_content(model_key, messages, temperature=0, max_tokens=MAX_NEW_TOKENS, json_mode=False):
    if model_key == "deepseek-flash":
        body = deepseek_chat(messages, temperature, max_tokens, json_mode)
        return body["choices"][0]["message"]["content"], body.get("usage", {})
    body = ollama_chat(
        model_key,
        messages,
        temperature=temperature,
        num_predict=max_tokens,
        response_format="json" if json_mode else None,
    )
    metrics = {
        key: body.get(key)
        for key in ["total_duration", "load_duration", "prompt_eval_count", "prompt_eval_duration", "eval_count", "eval_duration"]
    }
    return body["message"]["content"], metrics


def clean_expansion(text):
    text = text.strip().strip('"\'“”')
    prefixes = r"^(tentu( saja)?|baik|oke|ok|berdasarkan dokumen|saya dapat membantu)[^\n.:]*[.:,]\s*"
    return re.sub(prefixes, "", text, flags=re.IGNORECASE).strip().strip('"\'“”')


def expand_query(question, model_key):
    if model_uses_system_role(model_key):
        messages = [
            {"role": "system", "content": QUERY_EXPANSION_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ]
    else:
        messages = [{
            "role": "user",
            "content": f"{QUERY_EXPANSION_SYSTEM_PROMPT}\n\n{question}",
        }]
    content, _ = model_content(
        model_key,
        messages,
        temperature=QUERY_EXPANSION_TEMPERATURE,
        max_tokens=QUERY_EXPANSION_MAX_TOKENS,
    )
    expanded = clean_expansion(content)
    return [expanded] if expanded else []


def format_context(scored_documents):
    blocks = []
    for index, (document, score) in enumerate(scored_documents, start=1):
        metadata = document.metadata
        header = f"[{index}] {metadata.get('document_title', '-')}"
        if metadata.get("section_id"):
            header += f" | {metadata['section_id']}"
        blocks.append(f"{header}\n{document.page_content}")
    return "\n\n".join(blocks)


def build_messages(question, scored_documents, model_key):
    context = format_context(scored_documents)
    if context:
        user_content = f"Konteks:\n{context}\n\nPertanyaan: {question}"
    else:
        user_content = f"Pertanyaan: {question}\n\n(Tidak ditemukan bagian dokumen yang cukup relevan.)"
    if model_uses_system_role(model_key):
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
    return [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_content}"}]


def parse_json_response(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:]
    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        return {}


def generate_citations(question, scored_documents, model_key):
    if not scored_documents:
        return {}
    source_block = "\n\n".join(
        f"[{index}] {document.page_content}"
        for index, (document, _) in enumerate(scored_documents, start=1)
    )
    user_content = f"Pertanyaan: {question}\n\n{source_block}"
    if model_uses_system_role(model_key):
        messages = [
            {"role": "system", "content": CITATION_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
    else:
        messages = [{"role": "user", "content": f"{CITATION_SYSTEM_PROMPT}\n\n{user_content}"}]
    content, _ = model_content(
        model_key,
        messages,
        temperature=0,
        max_tokens=CITATION_MAX_TOKENS,
        json_mode=True,
    )
    data = parse_json_response(content)
    return {
        str(item.get("index")): item.get("answer", "")
        for item in data.get("sources", [])
        if isinstance(item, dict) and item.get("index") is not None
    }


retrieval_cache = {}


def retrieve(question, model_key):
    cache_key = (question, model_key)
    if cache_key in retrieval_cache:
        return retrieval_cache[cache_key]
    started = time.perf_counter()
    expansions = expand_query(question, model_key)
    candidates = hybrid_search([question] + expansions)
    scored = rerank(question, candidates)
    result = {
        "expansions": expansions,
        "scored_documents": scored,
        "retrieval_s": time.perf_counter() - started,
        "candidate_count": len(candidates),
    }
    retrieval_cache[cache_key] = result
    return result


def serialize_sources(scored_documents):
    return [
        {
            "chunk_id": document.metadata.get("chunk_id"),
            "faiss_id": document.metadata.get("faiss_id"),
            "document_title": document.metadata.get("document_title"),
            "section_id": document.metadata.get("section_id"),
            "page_start": document.metadata.get("page_start"),
            "page_end": document.metadata.get("page_end"),
            "bps_url": document.metadata.get("bps_url"),
            "rerank_score": score,
        }
        for document, score in scored_documents
    ]


def generate_local_rag(question, model_key):
    retrieval = retrieve(question, model_key)
    scored_documents = retrieval["scored_documents"]
    citations = generate_citations(question, scored_documents, model_key)
    started = time.perf_counter()
    body = ollama_chat(model_key, build_messages(question, scored_documents, model_key))
    generation_s = time.perf_counter() - started
    return body["message"]["content"], {
        "retrieval_model": model_key,
        "query_expansions": retrieval["expansions"],
        "retrieved_context": format_context(scored_documents),
        "context_chunks": [document.page_content for document, _ in scored_documents],
        "retrieved_sources": serialize_sources(scored_documents),
        "citations": citations,
        "retrieval_s": retrieval["retrieval_s"],
        "generation_s": generation_s,
        "ollama_metrics": {
            key: body.get(key)
            for key in ["total_duration", "load_duration", "prompt_eval_count", "prompt_eval_duration", "eval_count", "eval_duration"]
        },
    }


def generate_deepseek_rag(question):
    model_key = "deepseek-flash"
    retrieval = retrieve(question, model_key)
    scored_documents = retrieval["scored_documents"]
    citations = generate_citations(question, scored_documents, model_key)
    messages = build_messages(question, scored_documents, model_key)
    started = time.perf_counter()
    answer, usage = model_content(model_key, messages)
    generation_s = time.perf_counter() - started
    return answer, {
        "retrieval_model": model_key,
        "query_expansions": retrieval["expansions"],
        "retrieved_context": format_context(scored_documents),
        "context_chunks": [document.page_content for document, _ in scored_documents],
        "retrieved_sources": serialize_sources(scored_documents),
        "citations": citations,
        "retrieval_s": retrieval["retrieval_s"],
        "generation_s": generation_s,
        "usage": usage,
    }

## 7. Jalankan RAG untuk seluruh skenario dengan checkpoint

In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


CHECKPOINT_DIR = OUTPUT_DIR / "scenario_generation_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
completed = []

for scenario in scenario_bank.to_dict("records"):
    for model_key in MODELS_FOR_SCENARIO:
        checkpoint_path = CHECKPOINT_DIR / f"{scenario['scenario_id']}__{model_key}.json"
        if checkpoint_path.exists():
            row = json.loads(checkpoint_path.read_text(encoding="utf-8"))
            if row.get("status") == "ok":
                completed.append(row)
                continue

        started = time.perf_counter()
        try:
            if model_key in MODEL_CONFIGS:
                answer, details = generate_local_rag(scenario["question"], model_key)
            else:
                answer, details = generate_deepseek_rag(scenario["question"])
            row = {
                **scenario,
                "model_key": model_key,
                "answer": answer,
                "retrieval_model": details["retrieval_model"],
                "query_expansions": details["query_expansions"],
                "retrieved_context": details["retrieved_context"],
                "context_chunks": details["context_chunks"],
                "retrieved_sources": details["retrieved_sources"],
                "citations": details["citations"],
                "retrieval_s": details["retrieval_s"],
                "generation_s": details["generation_s"],
                "generation_meta": details.get("ollama_metrics", details.get("usage", {})),
                "wall_s": time.perf_counter() - started,
                "status": "ok",
                "error": "",
            }
        except Exception as exc:
            row = {
                **scenario,
                "model_key": model_key,
                "answer": "",
                "retrieval_model": "",
                "query_expansions": [],
                "retrieved_context": "",
                "context_chunks": [],
                "retrieved_sources": [],
                "citations": {},
                "retrieval_s": None,
                "generation_s": None,
                "generation_meta": {},
                "wall_s": time.perf_counter() - started,
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}",
            }
        write_json(checkpoint_path, row)
        completed.append(row)
        print(scenario["scenario_id"], model_key, row["status"])

scenario_df = pd.DataFrame(completed)
failures = scenario_df[scenario_df["status"] != "ok"]
if not failures.empty:
    display(failures[["scenario_id", "model_key", "error"]])
    raise RuntimeError("Ada pipeline RAG yang gagal. Perbaiki error lalu jalankan ulang.")

expected = len(scenario_bank) * len(MODELS_FOR_SCENARIO)
if len(scenario_df) != expected or scenario_df.duplicated(["scenario_id", "model_key"]).any():
    raise ValueError(f"Hasil harus berjumlah {expected} pasangan unik.")
display(scenario_df.groupby(["scenario", "model_key"]).size().unstack(fill_value=0))


## 8. Penilaian RAGAS sebelum penyimpanan CSV

Tiga metrik mengikuti notebook evaluasi RAG sebelumnya:

1. `faithfulness`: dukungan klaim jawaban oleh konteks retrieval.
2. `answer_relevancy`: relevansi jawaban terhadap pertanyaan.
3. `context_relevance`: relevansi konteks retrieval terhadap pertanyaan.

RAGAS menggunakan OpenAI sebagai judge, sedangkan model yang diuji tetap tiga model pada pipeline di atas. Jawaban acuan dan ID chunk sumber tidak digunakan sebagai kebenaran evaluasi. Setiap metrik dicache terpisah; error atau NaN dicatat dan tidak diubah menjadi nol.


In [ ]:
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def strip_quotes_for_ragas(text):
    text = re.sub(r"\[\d+\]", "", text)
    return text.replace('\"', '').replace('“', '').replace('”', '').strip()


def repetition_ratio(text):
    if not isinstance(text, str) or len(text) < 50:
        return 0.0
    compressed = re.sub(r"(.{1,15}?)\1{4,}", r"\1", text)
    return 1 - (len(compressed) / len(text))


DEGENERATE_THRESHOLD = 0.3
MIN_LEN_FOR_ANSWER_RELEVANCY = 15


def make_ragas_evaluators():
    from openai import AsyncOpenAI
    from ragas.embeddings import OpenAIEmbeddings
    from ragas.llms import llm_factory
    from ragas.metrics.collections import AnswerRelevancy, ContextRelevance, Faithfulness

    client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    judge = llm_factory(JUDGE_MODEL, client=client, max_tokens=4096)
    embedding = OpenAIEmbeddings(client=client, model=JUDGE_EMBEDDING)
    return {
        "faithfulness": Faithfulness(llm=judge),
        "answer_relevancy": AnswerRelevancy(llm=judge, embeddings=embedding),
        "context_relevance": ContextRelevance(llm=judge),
    }


async def evaluate_ragas_rows(frame):
    import asyncio

    evaluator_config = {
        "implementation": "ragas_0_4_collections_openai_three_metrics_v1",
        "judge": JUDGE_MODEL,
        "embedding": JUDGE_EMBEDDING,
        "ragas_version": metadata.version("ragas"),
        "openai_version": metadata.version("openai"),
        "instructor_version": metadata.version("instructor"),
        "degenerate_threshold": DEGENERATE_THRESHOLD,
        "min_len_answer_relevancy": MIN_LEN_FOR_ANSWER_RELEVANCY,
    }
    evaluator_id = digest(evaluator_config)
    write_json(OUTPUT_DIR / "ragas_evaluator.json", {**evaluator_config, "evaluator_id": evaluator_id})
    metrics = make_ragas_evaluators()
    rows = []

    for item in frame.to_dict("records"):
        contexts = item["context_chunks"]
        answer = item["answer"] if isinstance(item["answer"], str) else ""
        answer_clean = strip_quotes_for_ragas(answer)
        scored = {
            "ragas_evaluator_id": evaluator_id,
            "ragas_answer_degenerate": repetition_ratio(answer) > DEGENERATE_THRESHOLD,
        }
        if not answer.strip() or scored["ragas_answer_degenerate"]:
            status = "empty_answer" if not answer.strip() else "degenerate_answer"
            for name in metrics:
                scored[name] = np.nan
                scored[name + "_status"] = status
            rows.append(scored)
            continue

        for name, metric in metrics.items():
            if name == "answer_relevancy" and len(answer_clean) < MIN_LEN_FOR_ANSWER_RELEVANCY:
                scored[name] = np.nan
                scored[name + "_status"] = "too_short_for_answer_relevancy"
                continue
            if name != "answer_relevancy" and not contexts:
                scored[name] = np.nan
                scored[name + "_status"] = "empty_retrieval"
                continue

            inputs = {"user_input": item["question"]}
            if name != "context_relevance":
                inputs["response"] = answer_clean if name == "answer_relevancy" else answer
            if name != "answer_relevancy":
                inputs["retrieved_contexts"] = contexts

            cache_key = digest({"evaluator": evaluator_id, "metric": name, "inputs": inputs})
            cache_path = OUTPUT_DIR / "ragas_cache" / f"{cache_key}.json"
            cached = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
            if cached.get("status") != "ok":
                try:
                    result = await asyncio.wait_for(metric.ascore(**inputs), timeout=600)
                    value = float(result.value)
                    cached = {
                        "value": value if np.isfinite(value) else None,
                        "status": "ok" if np.isfinite(value) else "judge_nan",
                    }
                except Exception as exc:
                    cached = {"value": None, "status": "error_" + type(exc).__name__}
                write_json(cache_path, cached)
            scored[name] = np.nan if cached["value"] is None else cached["value"]
            scored[name + "_status"] = cached["status"]
        rows.append(scored)
    return pd.DataFrame(rows)


if RUN_RAGAS:
    ragas_scores = await evaluate_ragas_rows(scenario_df)
    scenario_df = pd.concat([scenario_df.reset_index(drop=True), ragas_scores], axis=1)
else:
    for metric_name in ["faithfulness", "answer_relevancy", "context_relevance"]:
        scenario_df[metric_name] = np.nan
        scenario_df[metric_name + "_status"] = "not_run"

result_path = OUTPUT_DIR / "scenario_results.csv"
csv_results = scenario_df.copy()
for column in ["query_expansions", "context_chunks", "retrieved_sources", "citations", "generation_meta"]:
    csv_results[column] = csv_results[column].map(lambda value: json.dumps(value, ensure_ascii=False))
csv_results.to_csv(result_path, index=False)

RAGAS_METRICS = ["faithfulness", "answer_relevancy", "context_relevance"]


def summarize_ragas(frame, groups):
    summary = frame.groupby(groups, dropna=False)[RAGAS_METRICS].agg(["mean", "count", "std", "median"])
    summary.columns = ["_".join(part for part in column if part) for column in summary.columns]
    return summary.reset_index()


ragas_summary_by_model = summarize_ragas(scenario_df, ["model_key"])
ragas_summary_by_scenario = summarize_ragas(scenario_df, ["model_key", "scenario"])

coverage_rows = []
for metric_name in RAGAS_METRICS:
    status_column = metric_name + "_status"
    for (model_key, scenario, status), count in scenario_df.groupby(
        ["model_key", "scenario", status_column]
    ).size().items():
        coverage_rows.append({
            "model_key": model_key,
            "scenario": scenario,
            "metric": metric_name,
            "status": status,
            "n": int(count),
        })
ragas_coverage = pd.DataFrame(coverage_rows)

# Variasi gaya dibandingkan dengan redaksi formal untuk pertanyaan yang sama.
style_ragas = scenario_df[scenario_df["scenario"].eq("3_variasi_gaya_bahasa")].copy()
formal = style_ragas[style_ragas["style"].eq("formal")][
    ["style_group_id", "model_key", *RAGAS_METRICS]
].rename(columns={metric: f"formal_{metric}" for metric in RAGAS_METRICS})
style_ragas = style_ragas.merge(formal, on=["style_group_id", "model_key"], validate="many_to_one")
for metric_name in RAGAS_METRICS:
    style_ragas[f"delta_{metric_name}_vs_formal"] = (
        style_ragas[metric_name] - style_ragas[f"formal_{metric_name}"]
    )
style_columns = [
    "scenario_id", "model_key", "variant", "style", "question", "style_group_id",
    *RAGAS_METRICS,
    *[f"delta_{metric}_vs_formal" for metric in RAGAS_METRICS],
]
style_ragas_output = style_ragas[style_columns]

# Pertanyaan typo dibandingkan dengan pasangan faktual bersihnya.
clean = scenario_df[scenario_df["scenario"].eq("1_faktual_berbasis_narasi")][
    ["base_question_id", "model_key", *RAGAS_METRICS]
].rename(columns={metric: f"clean_{metric}" for metric in RAGAS_METRICS})
typo_ragas = scenario_df[scenario_df["scenario"].eq("5_kesalahan_ketik")].merge(
    clean,
    on=["base_question_id", "model_key"],
    validate="one_to_one",
)
for metric_name in RAGAS_METRICS:
    typo_ragas[f"delta_{metric_name}_vs_clean"] = (
        typo_ragas[metric_name] - typo_ragas[f"clean_{metric_name}"]
    )
typo_columns = [
    "scenario_id", "model_key", "variant", "typo_severity", "typo_operations",
    "question", "base_question_id", *RAGAS_METRICS,
    *[f"delta_{metric}_vs_clean" for metric in RAGAS_METRICS],
]
typo_ragas_output = typo_ragas[typo_columns]

ragas_frames = {
    "scenario_ragas_summary_by_model.csv": ragas_summary_by_model,
    "scenario_ragas_summary_by_scenario.csv": ragas_summary_by_scenario,
    "scenario_ragas_coverage.csv": ragas_coverage,
    "scenario_ragas_style_comparison.csv": style_ragas_output,
    "scenario_ragas_typo_comparison.csv": typo_ragas_output,
}
ragas_output_paths = []
for filename, frame in ragas_frames.items():
    output_path = OUTPUT_DIR / filename
    frame.to_csv(output_path, index=False)
    ragas_output_paths.append(output_path)

display(ragas_summary_by_scenario)
display(ragas_coverage)
print("CSV prediksi dan seluruh keluaran RAGAS selesai disimpan.")


## 9. Buat tiga lembar evaluasi buta


In [ ]:
ordered = scenario_df.sort_values(["scenario_id", "model_key"]).reset_index(drop=True)
ordered["eval_id"] = [
    "E" + hashlib.sha256(f"{row.scenario_id}|{row.model_key}".encode()).hexdigest()[:11]
    for row in ordered.itertuples()
]

key_columns = [
    "eval_id", "scenario_id", "scenario", "variant", "model_key",
    "source_subtask_type", "base_question_id",
    "style_group_id", "style", "typo_group_id", "typo_operations",
    "typo_severity",
]
key_df = ordered[key_columns].copy()
key_path = OUTPUT_DIR / "scenario_human_eval_key.csv"
key_df.to_csv(key_path, index=False)

blind_columns = [
    "eval_id", "scenario", "variant", "typo_severity",
    "question", "retrieved_context", "answer",
]
packet = ordered[blind_columns].copy()
for column in ["fluency", "factual_correctness", "completeness", "notes"]:
    packet[column] = ""

evaluator_paths = []
for index, evaluator_name in enumerate(RATER_NAMES, start=1):
    evaluator_packet = packet.sample(frac=1, random_state=SEED + index).reset_index(drop=True)
    evaluator_packet.insert(0, "item_no", range(1, len(evaluator_packet) + 1))
    path = OUTPUT_DIR / f"scenario_human_eval_evaluator{index}.csv"
    evaluator_packet.to_csv(path, index=False)
    evaluator_paths.append(path)
    print(f"{evaluator_name}: {path.name} ({len(evaluator_packet)} item)")

rubric = '''Skala 1–5.
Fluency: 5 sangat lancar; 4 kesalahan kecil; 3 cukup dipahami; 2 sulit dipahami; 1 tidak dapat dipahami.
Factual correctness: 5 seluruh klaim benar; 4 kesalahan minor; 3 sebagian benar; 2 mayoritas keliru; 1 tidak benar.
Completeness: 5 seluruh kebutuhan terjawab; 4 hampir seluruhnya; 3 sebagian; 2 sangat terbatas; 1 tidak menjawab.
Untuk skenario di luar konteks publikasi BPS, penolakan yang tepat dinilai baik dan fakta karangan dinilai buruk.
Nilai factual correctness terhadap retrieved_context; jangan memakai pengetahuan di luar konteks sebagai pembenaran jawaban.
Jangan menebak identitas model dan jangan mengubah eval_id.
Setelah diisi, simpan sebagai scenario_human_eval_evaluatorN_filled.csv.
'''
rubric_path = OUTPUT_DIR / "scenario_human_eval_instructions.txt"
rubric_path.write_text(rubric, encoding="utf-8")

## 10. Unggah artefak ke Hugging Face


In [ ]:
manifest = {
    "scenario_source": "manual static question bank; no test.csv and no LLM question generation",
    "pipeline": "active model expansion -> dense+BM25 -> RRF -> reranker -> active model citation -> active model answer",
    "kb_repo": HF_KB_REPO,
    "asset_revisions": ASSET_REVISIONS,
    "models": model_manifest,
    "model_keys": MODELS_FOR_SCENARIO,
    "model_specific_expansion_and_citation": True,
    "evaluation": {
        "human": {"evaluators": RATER_NAMES, "dimensions": ["fluency", "factual_correctness", "completeness"]},
        "ragas": {"enabled": RUN_RAGAS, "metrics": RAGAS_METRICS, "judge": JUDGE_MODEL, "embedding": JUDGE_EMBEDDING},
    },
    "scenario_design": {
        "total": 20,
        "per_scenario": 4,
        "style": {"one_meaning": "IPM trend", "variants": 4, "formal_baseline": True},
        "typo": {"minor": 2, "major": 2, "paired_with_clean_factual": True},
    },
    "n_scenarios": len(scenario_bank),
    "n_outputs": len(scenario_df),
    "evaluator_ids": RATER_NAMES,
    "seed": SEED,
    "top_k": TOP_K,
    "num_ctx": NUM_CTX,
    "max_new_tokens": MAX_NEW_TOKENS,
    "note": "All 20 questions are manually frozen. LLMs do not create or rewrite evaluation questions.",
}
manifest_path = OUTPUT_DIR / "scenario_generation_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

api = HfApi(token=HF_TOKEN)
ragas_evaluator_path = OUTPUT_DIR / "ragas_evaluator.json"
optional_ragas_paths = [ragas_evaluator_path] if ragas_evaluator_path.exists() else []
upload_paths = [
    result_path, key_path, rubric_path, manifest_path,
    *optional_ragas_paths, *ragas_output_paths, *evaluator_paths,
]
for artifact_path in upload_paths:
    api.upload_file(
        path_or_fileobj=str(artifact_path),
        path_in_repo=artifact_path.name,
        repo_id=HF_REPO,
        repo_type="dataset",
    )
    print(f"Terunggah: {HF_REPO}/{artifact_path.name}")

print("Bagikan hanya tiga file evaluator dan petunjuk. Jangan bagikan scenario_human_eval_key.csv.")

## 11. Hentikan service runtime


In [ ]:
ollama_proc.terminate()
try:
    ollama_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    ollama_proc.kill()
print("Ollama runtime dihentikan.")